# **단순 예측 기준선**

복잡한 모델의 가치가 있는지는 **지난주 같은 요일 판매량**이나 **최근 4주 같은 요일 평균**보다 나은지 봐야 알 수 있습니다. 이 노트북은 두 기준선과 LightGBM을 같은 검증 행에서 비교합니다.

예측 기준일 `t`에서 1~7일 뒤의 판매량을 예측하며, 기준선에 쓰는 판매 이력은 모두 `t` 이전에 관측된 값입니다.

## **환경 준비**

별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 실행합니다. Drive를 연결하고 `/content/drive/MyDrive/retail-demand-forecasting`을 작업 폴더로 사용합니다.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Seaborn과 나눔고딕 글꼴을 설치합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib


## **Library**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

## **Utils**


In [ ]:
# 0 판매가 많은 데이터의 로그 오차를 계산합니다.
def rmsle(actual, predicted):
    actual = np.clip(np.asarray(actual), 0, None)
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))

In [ ]:
# 판매량 단위의 평균 절대오차를 계산합니다.
def mae(actual, predicted):
    return np.mean(np.abs(np.asarray(actual) - np.asarray(predicted)))

In [ ]:
# 같은 검증 행에서 모델과 두 기준선을 평가합니다.
def evaluate_baselines(frame, columns):
    rows = []
    for name, column in columns.items():
        rows.append({"model": name, "rmsle": rmsle(frame["target_sales"], frame[column]), "mae": mae(frame["target_sales"], frame[column])})
    return pd.DataFrame(rows).set_index("model").sort_values("rmsle")

## **실행 설정**


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
PROJECT_ROOT = REPO
RUN_NAME = "full_history_time_split_fixed_i600"
OUTPUT_PATH = REPO / "outputs" / RUN_NAME

## **Baseline 선정 근거**

> **EDA 근거:** 전체 판매량의 7일 자기상관(0.826)이 1일(0.767)보다 높고, 추세를 제거한 뒤에도 7일 주기가 남았습니다. 따라서 같은 요일의 과거 판매량을 가장 단순한 비교 기준으로 둡니다. 자세한 분석은 `01_eda.ipynb`에 있습니다.

목표일을 `d = t+h` (`h=1~7`)라고 할 때 두 Baseline은 다음과 같습니다.

| Baseline | 예측값 | 채택 이유 |
|---|---|---|
| 전주 동일 요일 | 같은 매장·상품군의 `d−7` 판매량 | 가장 최근의 주간 패턴을 그대로 반영 |
| 최근 4주 동일 요일 평균 | `d−7`, `d−14`, `d−21`, `d−28` 판매량의 평균 | 특정 한 주의 급증·급감 영향을 완화 |

예를 들어 8월 1일 영업 종료 후 8월 8일(`h=7`)을 예측하면, 전주 같은 요일인 **8월 1일 판매량**을 씁니다. `h≤7`이므로 사용한 날짜는 모두 기준일 `t` 또는 그 이전입니다. 두 기준선은 주간 패턴만으로 얻을 수 있는 성능을 보여주며, 아래 LightGBM 예측이 이를 개선하는지 확인합니다.


In [ ]:
predictions_path = OUTPUT_PATH / "validation_predictions_lightgbm.csv"
predictions_path = predictions_path if predictions_path.exists() else OUTPUT_PATH / "validation_predictions.csv"
predictions = pd.read_csv(
    predictions_path,
    parse_dates=["date", "target_date"],
)

In [ ]:
baseline_columns = {
    "LightGBM": "prediction",
    "Previous-week daily profile": "sales_same_weekday_last_week",
    "Same-weekday 4-week mean": "sales_same_weekday_4w_mean",
}
baseline_scores = evaluate_baselines(predictions, baseline_columns)
baseline_view = baseline_scores.round(4)
baseline_view


In [ ]:
baseline_plot = baseline_scores.reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=baseline_plot, x="rmsle", y="model", ax=axes[0], color="steelblue")
sns.barplot(data=baseline_plot, x="mae", y="model", ax=axes[1], color="darkorange")
axes[0].set(title="RMSLE", xlabel="낮을수록 좋음", ylabel="")
axes[1].set(title="MAE", xlabel="낮을수록 좋음", ylabel="")
fig.tight_layout()
plt.show()

전체 성능뿐 아니라 `forecast_horizon`별 RMSLE와 MAE를 함께 확인합니다. 예측 거리가 길어질수록 오차가 증가하는지, 통합 LightGBM이 각 거리에서 주간 계절 기준선을 안정적으로 개선하는지가 핵심 판단 기준입니다.

In [ ]:
horizon_scores = []
for horizon, group in predictions.groupby("forecast_horizon"):
    for name, column in baseline_columns.items():
        horizon_scores.append({
            "forecast_horizon": horizon,
            "model": name,
            "rmsle": rmsle(group["target_sales"], group[column]),
            "mae": mae(group["target_sales"], group[column]),
        })
horizon_scores = pd.DataFrame(horizon_scores)
horizon_view = horizon_scores.pivot(index="forecast_horizon", columns="model", values=["rmsle", "mae"]).round(4)
horizon_view
